In [16]:
import pandas as pd

In [17]:
df = pd.read_excel("step3result copy 2.xlsx") 

In [18]:
categories = df['category'].unique()
user_question = "Does a diagnosis of a connective tissue disease contribute to a post-dural spinal puncture headache?"

In [19]:
from langchain.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_openai import AzureChatOpenAI
import os
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains import MapReduceDocumentsChain, ReduceDocumentsChain
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import DataFrameLoader

# remove this later
CHAT = AzureChatOpenAI(
    azure_endpoint="https://nlp-ai-svc.openai.azure.com/",
    openai_api_version="2024-02-15-preview",
    azure_deployment="ChatGPT16k",
    openai_api_type="azure",
    temperature=0,
    model_name="gpt-35-turbo-16",
    openai_api_key=os.environ.get("OPENAI_API_KEY")
)


# CATEGORIZATION_SUMMARY_SYSTEM_TEMPLATE = """I am considering a scoping reivew project for given category I want to understand how the existing literature guides. Write a single detailed summary based on the full texts of the article for each category."""

SUMMARIZE_CATEGORY_TEMPLATE = "I am working  on a scoping review to address this question: {question}\n\n Currently, I am summarizing articles by expert-defined categories. All of the article summaries below were assigned the category {category}. Write a single paragrph final summary of the following journal article summaries, focusing on my question. Use APA-style in-text citations throughout the paragraph. The article summaries are separated by '---'"

SUMMARIZE_ARTICLE_TEMPLATE = "I am working  on a scoping review to address this question: {question}\n\n Currently, I am summarizing articles by expert-defined categories. The article below was assigned the category {category}. Write a single paragrph summarizing the artcle, focusing on my question. The entire paragraph will be cited using only the article being summarized. Write the paragaraph so providing only the citation for the single article will be appropriate."



HUMAN_TEMPLATE = """
Content to summarize:
{content}
"""

category_system_message_prompt = SystemMessagePromptTemplate.from_template(SUMMARIZE_CATEGORY_TEMPLATE)
article_system_message_prompt = SystemMessagePromptTemplate.from_template(SUMMARIZE_ARTICLE_TEMPLATE)

human_message_prompt = HumanMessagePromptTemplate.from_template(HUMAN_TEMPLATE)

category_summary_chat_prompt = ChatPromptTemplate.from_messages([category_system_message_prompt, human_message_prompt])
article_summary_chat_prompt = ChatPromptTemplate.from_messages([article_system_message_prompt, human_message_prompt])


In [20]:

output = []
for current_category in categories:
    filtered_rows = df[(df['category'] == current_category)]

    article_summaries = []
    for _, row in filtered_rows.iterrows():
        result = CHAT.invoke(article_summary_chat_prompt.format_prompt(
        question=user_question,
        category=current_category,
        content="APA Citation: " + row.APA_Citation + "\n\nArticle text:\n\n" +row.Text,
        ).to_messages())

        summary_to_keep = f"APA Citation: {row.APA_Citation}\n\n Summary: {result.content}\n\n --- "
        # print(summary_to_keep)
        article_summaries.append(summary_to_keep)

    text_to_summarize = "\n\n".join(article_summaries)
   # text_to_summarize = f"Category of current articles: {current_category} \n\n" + text_to_summarize
    
    result = CHAT.invoke(category_summary_chat_prompt.format_prompt(
        question=user_question,
        category=current_category,
        content=text_to_summarize
        ).to_messages())
    print(result.content)
    output.append(result.content)
    print("************")
    

The relationship between connective tissue diseases and post-dural spinal puncture headaches is complex and not fully understood. Wijayanayaka et al. (2021) reported a case of a patient with Alport syndrome, a connective tissue disease, who developed a postpartum headache and seizure following epidural anaesthesia, initially thought to be a postdural puncture headache. Similarly, Martin and Neilson (2014) suggested that joint hypermobility, a common symptom of connective tissue diseases, may contribute to the development of post-dural puncture headaches due to the increased flexibility and fragility of the dura mater. However, Golfinos et al. (1995) presented a case study of a patient with a history of connective tissue diseases, but found no direct evidence linking her condition to the development of a post-dural spinal puncture headache. Castori et al. (2015) and Chen and Tang (2022) provided valuable insights into the complex interplay between various types of headaches and connecti

In [ ]:
# use abtract when text is not available.
df['Text'] = df.apply(lambda row: row['abstract'] if row['Text'] == 'Text not available' else row['Text'], axis=1)


In [15]:
# Map reduce
for current_category in categories:
    # Map
    map_template = "I am working  on a scoping literature review to address this question: " + user_question + \
    "\n\nCurrently, I am summarizing articles by expert-defined categories. All of the articles below were assigned the category " + current_category + \
    """{docs}
    \n\nSummarize the provided articles, focusing on my question. Use in-text citations in APA format. The documents' metadata contains the full APA citation. 
    Use only the context of the articles, not any other information. \n\n"""
    map_prompt = PromptTemplate.from_template(map_template)
    map_chain = LLMChain(llm=CHAT, prompt=map_prompt)

    reduce_template = "I am working  on a scoping literature review to address this question: " + user_question + \
    "\n\nCurrently, I need a final summary of all the documents below. All of the article summaries below were assigned the category " + current_category + \
    """The following is set of summaries of articles:
    {docs}
    Take these and distill it into a final, consolidated summary paragraph appropriate for insertion into an academic journal article. Maintain in-text citations in the APA format. I'll add the full bibliography later. Use only the context of the articles, not any other information."""
    reduce_prompt = PromptTemplate.from_template(reduce_template)
    reduce_chain = LLMChain(llm=CHAT, prompt=reduce_prompt)

    # Takes a list of documents, combines them into a single string, and passes this to an LLMChain
    combine_documents_chain = StuffDocumentsChain(
        llm_chain=reduce_chain, document_variable_name="docs"
    )

    # Combines and iteratively reduces the mapped documents
    reduce_documents_chain = ReduceDocumentsChain(
        # This is final chain that is called.
        combine_documents_chain=combine_documents_chain,
        # If documents exceed context for `StuffDocumentsChain`
        collapse_documents_chain=combine_documents_chain,
        # The maximum number of tokens to group documents into.
        token_max=10000,
    )

    # Combining documents by mapping a chain over them, then combining results
    map_reduce_chain = MapReduceDocumentsChain(
        # Map chain
        llm_chain=map_chain,
        # Reduce chain
        reduce_documents_chain=reduce_documents_chain,
        # The variable name in the llm_chain to put the documents in
        document_variable_name="docs",
        # Return the results of the map steps in the output
        return_intermediate_steps=False,
    )





    filtered_rows = df[(df['category'] == current_category) & (df['Text'] != "Text not available")]
    text_columns = df[['Text', 'APA_Citation']]
    loader = DataFrameLoader(text_columns, page_content_column="Text")
    docs = loader.load()
    # if not filtered_rows.empty:
    #     new_df = pd.concat([new_df, filtered_rows])
    text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100000, chunk_overlap=0
    )
    split_docs = text_splitter.split_documents(docs)
    print(map_reduce_chain.run(split_docs))


The scoping literature review aimed to address the question of whether a diagnosis of a connective tissue disease contributes to a post-dural spinal puncture headache. However, the provided articles did not directly address this question. Instead, they focused on various topics such as neurological disorders, chronic daily headache, Ehlers-Danlos Syndrome (EDS), intrathecal analgesia, chronic inflammatory demyelinating polyradiculoneuropathy (CIDP), systemic lupus erythematosus (SLE), and periodontitis. These articles provided insights into the phenotype of new daily persistent headache (NDPH) and its comparison to transformed chronic daily headache (T-CDH), the barriers faced by physicians in diagnosing and managing EDS, the use of intrathecal analgesia in a patient with progressive systemic sclerosis (PSS), the association between CIDP and concomitant diseases, the long-term safety and efficacy of anifrolumab in SLE patients, the association between SSc and periodontitis, the comorbi